In [ ]:
import pandas as pd
import numpy as np 
import scanpy as sc
from sklearn.model_selection import train_test_split

## Read in data

You will need to have your `.h5ad` file to be able to split the data. 

Our formatting assumes that the file was outputted by running `process_multisample_TCR.ipynb`. If you have your own `.h5ad` file please take care to ensure that the labels align.

In [ ]:
# adata saved in .h5ad file or otherwise defined
adata = sc.read("/PATH/TO/your_tcr_only_object.h5ad")
adata.obs.head()

In [ ]:
len(adata.obs)

In [ ]:
len(adata.obs['barcode_t'].unique())

In [ ]:
adata.obs['cell_idx'] = adata.obs.index
adata.obs[['cell_idx', 'barcode_t', 'v_gene_t_tra']].head()

In [ ]:
adata.obs[['cell_idx', 'barcode_t', 'sample_id', 'label', 'label_encoded']].head()

In [ ]:
# hyperparams
save_dir = "hand_stratified_data"
save_dir_subdir = "testing-testing"
seed_id = 42

In [ ]:
adata.obs['label'].unique()

## Data Splitting and Stratification

Before any model training or evaluation, we split the dataset into training, validation, and test sets at the **patient level** to ensure that no patient appears in more than one split. This preserves independence between splits and prevents data leakage.

Given the limited number of patients and varying numbers of cells per patient, we performed a **custom stratification** to ensure that each split contained at least one patient from each diagnosis category: healthy (0), Non-Sjogren’s disease (1), and Primary Sjogren’s disease (2). Furthermore, to approximately match split ratios (that can be specified and changed as needed), patient assignments to train/validation/test were weighted by the number of cells per patient. This ensured that each split contained roughly the intended number of cells while maintaining patient-level separation and representation of all diagnosis categories. This approach was used solely for data organization and does not involve any access to model outputs or performance metrics. 

The ratios we ended up using for splits were approximately:

- Training: 70%  
- Validation: 15%  
- Test: 15%  

Due to constraints from the small number of patients, exact proportions and diagnosis distributions could not be perfectly maintained. However, this approach ensures that every split contains representation from all diagnosis categories while maintaining patient-level separation.

No metrics from any models were used to guide this split; this step is entirely independent of model training and evaluation (and was ran before it).

In [ ]:
np.random.seed(seed_id)  # reproducibility

In [ ]:
adata_labeled = adata.obs.copy()

In [ ]:
# compute cell counts per patient
cell_counts = adata_labeled.groupby('sample_id').size().reset_index(name='cell_count')

# get unique patient rows
patients = (
    adata_labeled[['sample_id', 'label_encoded']]
    .drop_duplicates()
    .merge(cell_counts, on='sample_id', how='left')
)

In [ ]:
# what you want to name your csv files
train_dataset_name = "trainset"
test_dataset_name = "testset"
val_dataset_name = "valset"

train_dataset_info_name = "trainset_info"
test_dataset_info_name = "testset_info"
val_dataset_info_name = "valset_info"

In [ ]:
groups = patients.groupby('label_encoded')
train_ratio = 0.7
val_ratio   = 0.15
test_ratio  = 0.15

assert abs(train_ratio + val_ratio + test_ratio - 1) < 1e-7

#train_count = round(train_ratio * total_cells_in_label_g)
#val_count   = round(val_ratio   * total_cells_in_label_g)
#test_count  = round(test_ratio  * total_cells_in_label_g)

In [ ]:
def split_group_weighted(df_group, train_ratio, val_ratio, test_ratio, seed=42):
    """Split ONE label group into train/val/test using cell-weighted tiebreaking."""
    np.random.seed(seed)
    rng = np.random.default_rng(seed)
    df = df_group.copy()

    # Add randomness for tie-breaking (and near-ties)
    if seed is not None:
        df['noise'] = rng.random(len(df))
        df = df.sort_values(['cell_count', 'noise'], ascending=[False, True]).drop(columns='noise')
    else:
        df = df.sort_values('cell_count', ascending=False)

    total_cells = df['cell_count'].sum()

    target_train = train_ratio * total_cells
    target_val   = val_ratio   * total_cells
    target_test  = test_ratio  * total_cells

    splits = {'train': [], 'val': [], 'test': []}
    counts = {'train': 0, 'val': 0, 'test': 0}

    for _, row in df.iterrows():
        pid = row['sample_id']
        cells = row['cell_count']

        deficits = {
            'train': target_train - counts['train'],
            'val':   target_val   - counts['val'],
            'test':  target_test  - counts['test']
        }

        best_split = max(deficits, key=deficits.get)

        splits[best_split].append(pid)
        counts[best_split] += cells

    return splits

In [ ]:
train_ids = []
val_ids   = []
test_ids  = []

for label, group in groups:
    group_splits = split_group_weighted(group, train_ratio, val_ratio, test_ratio)
    train_ids.extend(group_splits['train'])
    val_ids.extend(group_splits['val'])
    test_ids.extend(group_splits['test'])

In [ ]:
train_df = adata_labeled[adata_labeled['sample_id'].isin(train_ids)].copy()
val_df   = adata_labeled[adata_labeled['sample_id'].isin(val_ids)].copy()
test_df  = adata_labeled[adata_labeled['sample_id'].isin(test_ids)].copy()

patient_info_unique = adata_labeled[['sample_id', 'label', 'label_encoded']].drop_duplicates()

train_summary = patient_info_unique[
    patient_info_unique['sample_id'].isin(train_ids)
]

val_summary = patient_info_unique[
    patient_info_unique['sample_id'].isin(val_ids)
]

test_summary = patient_info_unique[
    patient_info_unique['sample_id'].isin(test_ids)
]

In [ ]:
def check_split(df, name):
    total = len(df)
    print(f"{name}: {total} cells")

check_split(train_df, "train")
check_split(val_df, "val")
check_split(test_df, "test")

In [ ]:
test_summary

## Save out

In [ ]:
import os

print(save_dir+"/"+save_dir_subdir)
os.makedirs(save_dir+"/"+save_dir_subdir, exist_ok=True)

In [ ]:
# save patient id breakdown info files
train_summary.to_csv(save_dir+"/"+save_dir_subdir+"/"+train_dataset_info_name+".csv", index=False)
test_summary.to_csv(save_dir+"/"+save_dir_subdir+"/"+test_dataset_info_name+".csv", index=False)
val_summary.to_csv(save_dir+"/"+save_dir_subdir+"/"+val_dataset_info_name+".csv", index=False)

# save the data itself
train_df.to_csv(save_dir+"/"+save_dir_subdir+"/"+train_dataset_name+".csv", index=False)
test_df.to_csv(save_dir+"/"+save_dir_subdir+"/"+test_dataset_name+".csv", index=False)
val_df.to_csv(save_dir+"/"+save_dir_subdir+"/"+val_dataset_name+".csv", index=False)